# EDA 및 변형 가능성 사전 선별 (연구계획서 5.4절, 5.7절)

이 노트북은 `data/raw/`에 내려받은 TDC ADMET Benchmark Group 22종 데이터셋을 열어 기초 EDA를 수행하고, 착수 게이트에 필요한 카운트(호변이성질체·양성자화 상태·염 형태·입체 표기)를 산출한다.

순서는 연구계획서 5.7절을 따른다: SMILES 파싱 검증 -> 부모 분자 확정(원본 보존) -> 정준화 및 중복 확인 -> 골격 계산. 여기까지는 데이터셋 전체에 대한 점검이므로 train_val/test를 합쳐서 봐도 분할 규율을 어기지 않는다. 실제 학습/시험 분할과 변형 생성은 이 노트북의 범위가 아니다.

로컬에서 이미 `data/raw/`를 만들어 두었다면 셀 2, 3은 건너뛰어도 된다. Colab에서 새로 시작하는 경우 순서대로 실행한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_ROOT = '/content/drive/MyDrive/Conference_2026'
assert os.path.isdir(PROJECT_ROOT), (
    f"{PROJECT_ROOT} 를 찾을 수 없다. Drive에서 폴더 이름이 다르면 이 경로를 직접 수정한다."
)
print('project root:', PROJECT_ROOT)

In [ ]:
# CLAUDE.md 방침: Colab에서는 로컬 venv를 쓰지 않고 노트북 안에서 그때그때 설치한다.
!pip install -q rdkit dimorphite-dl

In [ ]:
import os

RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')

if not os.path.isdir(RAW_DIR) or not os.listdir(RAW_DIR):
    print('data/raw가 비어 있다. 데이터셋을 먼저 내려받는다.')
    %cd {PROJECT_ROOT}
    !python scripts/download_admet.py
    !python scripts/download_esol_pilot.py
else:
    print('data/raw 확인됨:', sorted(os.listdir(RAW_DIR))[:5], '...')

In [ ]:
import sys

sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from scripts.eda_and_prescreen import load_dataset, profile_dataset, RAW_DIR as SCRIPT_RAW_DIR

print('스크립트가 보는 data/raw 경로:', SCRIPT_RAW_DIR)

## 1. 기초 EDA (데이터셋 하나 예시: bbb_martins)

다른 데이터셋을 보려면 `NAME`만 바꾼다. 열은 항상 `Drug_ID, Drug(SMILES), Y(라벨)` 셋뿐이다.

In [ ]:
NAME = 'bbb_martins'

df = load_dataset(NAME)
print('shape:', df.shape)
print('결측치:\n', df.isna().sum())
df.head()

In [ ]:
# 라벨 분포. 분류 물성이면 클래스 비율, 회귀 물성이면 기술통계를 본다.
if df['Y'].nunique() <= 10:
    print(df['Y'].value_counts(normalize=True))
else:
    print(df['Y'].describe())

In [ ]:
# SMILES 길이 분포 -- 이상치(너무 길거나 짧은 문자열)를 훑어본다.
df['smiles_len'] = df['Drug'].str.len()
print(df['smiles_len'].describe())
df.nlargest(5, 'smiles_len')[['Drug_ID', 'Drug', 'smiles_len']]

## 2. 착수 게이트: 22종 전체 사전 선별

5.4절 판정 대상 네 가지를 전부 계산한다. 데이터셋이 클수록(예: cyp2d6_veith 13,130개) 시간이 걸린다. 분자당 약 20~30ms이므로 22종 전체는 대략 20~40분 정도 예상한다. 중간에 끊겨도 이미 처리한 데이터셋의 결과는 `data/processed/<name>/molecule_profile.csv`에 저장되어 있으니, 아래 셀은 처리 안 된 데이터셋만 다시 돌리도록 짜여 있다.

In [ ]:
import time
from pathlib import Path

PROCESSED_DIR = Path(PROJECT_ROOT) / 'data' / 'processed'
RAW_DIR_PATH = Path(PROJECT_ROOT) / 'data' / 'raw'

names = sorted(
    p.name for p in RAW_DIR_PATH.iterdir()
    if p.is_dir() and p.name not in ('_tdc_cache', 'esol_pilot')
)

summaries = []
for name in names:
    cache_path = PROCESSED_DIR / name / 'molecule_profile.csv'
    if cache_path.exists():
        print(f'[skip, cached] {name}')
        continue
    t0 = time.time()
    s = profile_dataset(name)
    s['seconds'] = round(time.time() - t0, 1)
    summaries.append(s)
    print(f"[{s['seconds']:>6.1f}s] {name:35s} n={s['n_total']:>6d} "
          f"salt={s['pct_with_salt']}%  stereo={s['pct_with_stereo']}%  "
          f"multi-taut={s['pct_multi_tautomer']}%  multi-proto={s['pct_multi_protomer']}%")

print('\n새로 처리한 데이터셋:', len(summaries))

## 3. 게이트 판정

5.4절 기준: 해당 축의 변형 가능 분자가 전체의 10퍼센트 미만이면 그 축은 주 분석에서 제외하고 보조 관찰로만 남긴다.

In [ ]:
import json

rows = []
for name in names:
    detail_path = PROCESSED_DIR / name / 'molecule_profile.csv'
    if not detail_path.exists():
        continue
    d = pd.read_csv(detail_path)
    valid = d[d['valid']]
    n = len(valid)
    rows.append({
        'dataset': name,
        'n_valid': n,
        'pct_multi_tautomer_or_protomer': round(
            100 * ((valid['n_tautomers'] >= 2) | (valid['n_protomers'] >= 2)).mean(), 1
        ) if n else None,
        'pct_with_salt': round(100 * valid['has_salt'].mean(), 1) if n else None,
        'pct_with_stereo': round(100 * valid['has_stereo'].mean(), 1) if n else None,
    })

gate_df = pd.DataFrame(rows).sort_values('dataset')
gate_df['B1_생존'] = gate_df['pct_multi_tautomer_or_protomer'] >= 10
gate_df['B2_생존'] = gate_df['pct_with_salt'] >= 10
gate_df['B3_생존'] = gate_df['pct_with_stereo'] >= 10

gate_df.to_csv(PROCESSED_DIR / 'gate_decision.csv', index=False)
gate_df

In [ ]:
print('B1(미소상태) 생존 데이터셋 수:', gate_df['B1_생존'].sum(), '/', len(gate_df))
print('B2(염 형태) 생존 데이터셋 수:  ', gate_df['B2_생존'].sum(), '/', len(gate_df))
print('B3(입체 표기) 생존 데이터셋 수:', gate_df['B3_생존'].sum(), '/', len(gate_df))
print()
print('B2, B3 둘 다 죽은 데이터셋(주의 필요):')
print(gate_df[~gate_df['B2_생존'] & ~gate_df['B3_생존']]['dataset'].tolist())